In [ ]:
# Imports
import pandas as pd
import numpy as np
import os


['admissions.csv.gz', 'diagnoses_icd.csv.gz', 'drgcodes.csv.gz', 'd_hcpcs.csv.gz', 'd_icd_diagnoses.csv.gz', 'd_icd_procedures.csv.gz', 'd_labitems.csv.gz', 'emar.csv.gz', 'emar_detail.csv.gz', 'hcpcsevents.csv.gz', 'labevents.csv.gz', 'microbiologyevents.csv.gz', 'omr.csv.gz', 'patients.csv.gz', 'pharmacy.csv.gz', 'poe.csv.gz', 'poe_detail.csv.gz', 'prescriptions.csv.gz', 'procedures_icd.csv.gz', 'provider.csv.gz', 'services.csv.gz', 'transfers.csv.gz']


In [75]:
# Importing DB...
BASE_DB = "mimic-iv-clinical-database-demo-2.2"
HOSP_PATH = os.path.join(BASE_DB, "hosp")
ICU_PATH = os.path.join(BASE_DB, "icu")

# Loading Tables...
patients = pd.read_csv(f'{HOSP_PATH}/patients.csv.gz')
admissions = pd.read_csv(f'{HOSP_PATH}/admissions.csv.gz')
lab_events = pd.read_csv(f'{HOSP_PATH}/labevents.csv.gz')

chart_events = pd.read_csv(f'{ICU_PATH}/chartevents.csv.gz')
icu_stays = pd.read_csv(f'{ICU_PATH}/icustays.csv.gz') 

chart_events["warning"].unique()

array([ 0.,  1., nan])

# Step 1: Data Extraction | Feature Definition

In [ ]:
#convert to datetime
icu_stays['intime'] = pd.to_datetime(icu_stays['intime'])
icu_stays['outtime'] = pd.to_datetime(icu_stays['outtime'])

# Select first ICU stay per patient
icu_stays = icu_stays.sort_values(by="intime")
icu_stay_first = icu_stays.groupby("subject_id").first()

# add the patients' demographics based on the chosen icu stay
df = icu_stay_first.merge(patients, on="subject_id")

# add relevant hospital admission
df = df.merge(admissions, on=["subject_id", "hadm_id"])

# Define chartevents for 24H window (var:24H_endtime)
# Calculate the 24H_endtime based on each icu_stay intime
# Extract the chartevents done in those 24H
df["24H_endtime"] = df["intime"] + pd.Timedelta(hours=24)
#df = df.merge(chart_events, on=["subject_id","hadm_id", "stay_id"])
chart_events["charttime"] = pd.to_datetime(chart_events["charttime"])





# Keep relevant columns
df = df[[
     "subject_id","hadm_id", 
     "stay_id", "first_careunit",
     "anchor_age", "gender",
     "admission_type", "hospital_expire_flag",
     "intime","24H_endtime", "outtime"
]]


df.head()

0         2132-12-16 00:00:00
1         2132-12-16 00:00:00
2         2132-12-16 00:00:00
3         2132-12-16 00:00:00
4         2132-12-16 00:00:00
                 ...         
668857    2153-03-28 10:49:28
668858    2153-03-28 10:49:28
668859    2153-03-28 10:49:28
668860    2153-03-28 10:49:28
668861    2153-03-28 10:49:28
Name: charttime, Length: 668862, dtype: str


,subject_id,hadm_id,stay_id,first_careunit,anchor_age,gender,admission_type,hospital_expire_flag,intime,24H_endtime,outtime
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),52,F,EW EMER.,0,2180-07-23 14:00:00,2180-07-24 14:00:00,2180-07-23 23:50:47
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),55,F,EW EMER.,0,2157-11-20 19:18:02,2157-11-21 19:18:02,2157-11-21 22:08:00
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),46,F,EW EMER.,0,2110-04-11 15:52:22,2110-04-12 15:52:22,2110-04-12 23:59:56
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),80,F,EW EMER.,0,2156-04-12 16:24:18,2156-04-13 16:24:18,2156-04-17 15:57:08
4,10002495,24982426,36753294,Coronary Care Unit (CCU),81,M,URGENT,0,2141-05-22 20:18:01,2141-05-23 20:18:01,2141-05-27 22:24:02
